<a href="https://colab.research.google.com/github/carvatreek25/teste_2/blob/main/untitled9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
df_video = spark.read.parquet("videos-comments-tratados.snappy.parquet")

In [ ]:
df_video = df_video.withColumn("Month", month(col("Published At")))

In [ ]:
indexer = StringIndexer(inputCol="keyword", outputCol="Keyword Index")
df_video = indexer.fit(df_video).transform(df_video)

In [ ]:
from pyspark.sql.functions import col
# 4. Criar o vetor de Features
# Convert 'Year' column to integer type
df_video = df_video.withColumn("Year", col("Year").cast("int"))
input_cols = ["Likes", "Views", "Year", "Month", "Keyword Index"]
assembler = VectorAssembler(inputCols=input_cols, outputCol="Features")
df_video = assembler.transform(df_video)

In [ ]:
df_video = df_video.na.drop(subset=["Features"]) # Limpeza de nulos

In [ ]:
scaler = StandardScaler(inputCol="Features", outputCol="Features Normal", withStd=True, withMean=False)
df_video = scaler.fit(df_video).transform(df_video)

# Conferindo se as colunas novas surgiram
df_video.select("Features", "Features Normal").show(5)

+--------------------+--------------------+
|            Features|     Features Normal|
+--------------------+--------------------+
|[3407.0,135612.0,...|[0.00423827538001...|
|[3407.0,135612.0,...|[0.00423827538001...|
|[3407.0,135612.0,...|[0.00423827538001...|
|[3407.0,135612.0,...|[0.00423827538001...|
|[3407.0,135612.0,...|[0.00423827538001...|
+--------------------+--------------------+
only showing top 5 rows


In [ ]:
# 6. Reduzir de 5 características para 1 usando PCA
pca = PCA(k=1, inputCol="Features Normal", outputCol="Features PCA")

# Drop the 'Features PCA' column if it already exists
if "Features PCA" in df_video.columns:
    df_video = df_video.drop("Features PCA")

model_pca = pca.fit(df_video)
df_video = model_pca.transform(df_video)

In [ ]:
# 7. Separar em 80% para treino e 20% para teste
train_data, test_data = df_video.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# 8. Criar o modelo de Regressão Linear para estimar "Comments"
# Usaremos a coluna 'Features Normal' como solicitado no exercício
lr = LinearRegression(featuresCol="Features Normal", labelCol="Comments")
lr_model = lr.fit(train_data)



# Avaliando o modelo nos dados de teste
results = lr_model.evaluate(test_data)
print(f"Erro Quadrático Médio (RMSE): {results.rootMeanSquaredError}")

Erro Quadrático Médio (RMSE): 25370.33362187817


In [ ]:
# 9. Salvar o dataframe como parquet
df_video.write.mode("overwrite").parquet("videos-preparados-parquet")